In [2]:
from rich.jupyter import display

# ==========================================
# TASK 1 - LOAD DATASET & INSPECT INFORMATION
# ==========================================
print("=================================================================")
print("TASK 1: DATASET INSPECTION")
print("=================================================================")

import pandas as pd
import numpy as np

# Load the real and fake news datasets from CSV files
# (Make sure 'True.csv' and 'Fake.csv' are in your working directory)
df_true = pd.read_csv("True.csv")
df_false = pd.read_csv("Fake.csv")

# Print the dimensions (rows and columns) of each dataset
print(f"True News Shape (Rows, Columns): {df_true.shape}")
print(f"Fake News Shape (Rows, Columns): {df_false.shape}\n")

# Display general info and structure of the True dataset
print("=== TRUE NEWS DATASET INFO ===")
print(df_true.info())

print("\n=== FAKE NEWS DATASET INFO ===")
print(df_false.info())

TASK 1: DATASET INSPECTION
True News Shape (Rows, Columns): (21417, 4)
Fake News Shape (Rows, Columns): (23481, 4)

=== TRUE NEWS DATASET INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    21417 non-null  str  
 1   text     21417 non-null  str  
 2   subject  21417 non-null  str  
 3   date     21417 non-null  str  
dtypes: str(4)
memory usage: 669.4 KB
None

=== FAKE NEWS DATASET INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    23481 non-null  str  
 1   text     23481 non-null  str  
 2   subject  23481 non-null  str  
 3   date     23481 non-null  str  
dtypes: str(4)
memory usage: 733.9 KB
None


In [3]:
# ==========================================
# TASK 2 - CHECK CLASS BALANCE & CREATE LABEL
# ==========================================
print("=================================================================")
print("TASK 2: CLASS BALANCE & LABEL CREATION")
print("=================================================================")

# Add a target label column to each dataframe
# 1 represents Real News, 0 represents Fake News
df_true['label'] = 1
df_false['label'] = 0

# Concatenate both dataframes into a single master dataset
df_combined = pd.concat([df_true, df_false], ignore_index=True)

# Shuffle the dataset randomly to prevent bias during training
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# Check class distribution to see if classes are balanced
class_counts = df_combined['label'].value_counts()
print("=== CLASS DISTRIBUTION ===")
print(class_counts)
print(f"\nPercentage of Real News (1): {(class_counts[1] / len(df_combined)) * 100:.2f}%")
print(f"Percentage of Fake News (0): {(class_counts[0] / len(df_combined)) * 100:.2f}%")

# Display the first few rows of our combined, labeled dataset
print("\n=== SAMPLE COMBINED DATA ===")
display(df_combined[['title', 'text', 'label']].head(3))

TASK 2: CLASS BALANCE & LABEL CREATION
=== CLASS DISTRIBUTION ===
label
0    23481
1    21417
Name: count, dtype: int64

Percentage of Real News (1): 47.70%
Percentage of Fake News (0): 52.30%

=== SAMPLE COMBINED DATA ===


,title,text,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",0
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,0


In [4]:
# ==========================================
# TASKS 3 & 4 - TRAIN-TEST SPLIT & SHAPES
# ==========================================
print("=================================================================")
print("TASKS 3 & 4: TRAIN-TEST SPLIT")
print("=================================================================")

from sklearn.model_selection import train_test_split

# Define features (text) and targets (labels)
X = df_combined['text']
y = df_combined['label']

# Task 3: Create a 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Task 4: Print the shapes of the resulting datasets
print(f"X_train shape (Training Texts):  {X_train.shape}")
print(f"X_test shape  (Testing Texts):   {X_test.shape}")
print(f"y_train shape (Training Labels): {y_train.shape}")
print(f"y_test shape  (Testing Labels):  {y_test.shape}")

TASKS 3 & 4: TRAIN-TEST SPLIT
X_train shape (Training Texts):  (35918,)
X_test shape  (Testing Texts):   (8980,)
y_train shape (Training Labels): (35918,)
y_test shape  (Testing Labels):  (8980,)


In [5]:
# ==========================================
# TASKS 5 & 6 - BUILD AND TRAIN PIPELINE
# ==========================================
print("=================================================================")
print("TASKS 5 & 6: TRIGRAM KNN PIPELINE TRAINING")
print("=================================================================")

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KNeighborsClassifier
import time

# Task 5: Create the Trigram (1,3) and KNN (10 neighbors, euclidean) pipeline
pipeline_knn_euclidean = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 3))),
    ('classifier', KNeighborsClassifier(n_neighbors=10, metric='euclidean'))
])

print("Pipeline created successfully. Starting training (this may take a few minutes)...")
start_time = time.time()

# Task 6: Train the pipeline using the training data
pipeline_knn_euclidean.fit(X_train, y_train)

print(f"Training completed in {time.time() - start_time:.2f} seconds!")

TASKS 5 & 6: TRIGRAM KNN PIPELINE TRAINING
Pipeline created successfully. Starting training (this may take a few minutes)...
Training completed in 71.25 seconds!


In [10]:
# ==========================================
# TASK 7 - CLASSIFICATION RESULTS
# ==========================================
print("=================================================================")
print("TASK 7: PREDICTION vs TARGET RESULTS")
print("=================================================================")

from sklearn.metrics import classification_report
import pandas as pd

# Generate predictions on the unseen test set
y_pred = pipeline_knn_euclidean.predict(X_test)

# Print the overall classification report (Accuracy, Precision, Recall)
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Fake (0)', 'Real (1)']))

# Create a clean DataFrame to visually compare predictions vs actual targets
print("\n=== COMPARISON: PREDICTIONS VS ACTUAL TARGETS ===")
comparison_df = pd.DataFrame({
    'Actual Target (y_test)': y_test.values,
    'Model Prediction (y_pred)': y_pred,
    'Correct Match?': y_test.values == y_pred
})

# Display the first 5 rows cleanly using display()
display(comparison_df.head(5))

TASK 7: PREDICTION vs TARGET RESULTS
=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

    Fake (0)       0.76      0.80      0.78      4669
    Real (1)       0.77      0.72      0.74      4311

    accuracy                           0.76      8980
   macro avg       0.76      0.76      0.76      8980
weighted avg       0.76      0.76      0.76      8980


=== COMPARISON: PREDICTIONS VS ACTUAL TARGETS ===


,Actual Target (y_test),Model Prediction (y_pred),Correct Match?
0,1,0,False
1,0,0,True
2,0,0,True
3,1,0,False
4,0,0,True


In [11]:
# ==========================================
# TASK 8 - TRIGRAM KNN (COSINE) PIPELINE
# ==========================================
print("=================================================================")
print("TASK 8: KNN PIPELINE WITH COSINE METRIC")
print("=================================================================")

# 1. Create the new pipeline using the 'cosine' metric
pipeline_knn_cosine = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 3))),
    ('classifier', KNeighborsClassifier(n_neighbors=10, metric='cosine'))
])

print("Training KNN (Cosine) model... (This may take a few minutes)")
start_time = time.time()

# 2. Train the pipeline
pipeline_knn_cosine.fit(X_train, y_train)
print(f"Training completed in {time.time() - start_time:.2f} seconds!\n")

# 3. Generate predictions and evaluate
y_pred_cosine = pipeline_knn_cosine.predict(X_test)

print("=== CLASSIFICATION REPORT (KNN - COSINE) ===")
print(classification_report(y_test, y_pred_cosine, target_names=['Fake (0)', 'Real (1)']))

TASK 8: KNN PIPELINE WITH COSINE METRIC
Training KNN (Cosine) model... (This may take a few minutes)
Training completed in 57.35 seconds!

=== CLASSIFICATION REPORT (KNN - COSINE) ===
              precision    recall  f1-score   support

    Fake (0)       0.64      0.98      0.77      4669
    Real (1)       0.96      0.39      0.56      4311

    accuracy                           0.70      8980
   macro avg       0.80      0.69      0.66      8980
weighted avg       0.79      0.70      0.67      8980



In [12]:
# ==========================================
# TASK 9 - RANDOM FOREST PIPELINE
# ==========================================
print("=================================================================")
print("TASK 9: RANDOM FOREST PIPELINE")
print("=================================================================")

from sklearn.ensemble import RandomForestClassifier

# 1. Create the new pipeline using RandomForest
pipeline_rf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 3))),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("Training Random Forest model... (This might take a while due to tree building)")
start_time = time.time()

# 2. Train the pipeline
pipeline_rf.fit(X_train, y_train)
print(f"Training completed in {time.time() - start_time:.2f} seconds!\n")

# 3. Generate predictions and evaluate
y_pred_rf = pipeline_rf.predict(X_test)

print("=== CLASSIFICATION REPORT (RANDOM FOREST) ===")
print(classification_report(y_test, y_pred_rf, target_names=['Fake (0)', 'Real (1)']))

TASK 9: RANDOM FOREST PIPELINE
Training Random Forest model... (This might take a while due to tree building)
Training completed in 1434.41 seconds!

=== CLASSIFICATION REPORT (RANDOM FOREST) ===
              precision    recall  f1-score   support

    Fake (0)       0.99      0.99      0.99      4669
    Real (1)       0.98      0.99      0.99      4311

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [13]:
# ==========================================
# TASK 10 - TRY OTHER PIPELINES (LOGISTIC REGRESSION)
# ==========================================
print("=================================================================")
print("TASK 10: LOGISTIC REGRESSION PIPELINE")
print("=================================================================")

from sklearn.linear_model import LogisticRegression
import time

# 1. Create a Logistic Regression pipeline
# We set max_iter=1000 to ensure the mathematical solver has enough time to converge
pipeline_lr = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1, 3))),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

print("Training Logistic Regression model... (This should be much faster!)")
start_time = time.time()

# 2. Train the pipeline
pipeline_lr.fit(X_train, y_train)
print(f"Training completed in {time.time() - start_time:.2f} seconds!\n")

# 3. Generate predictions and evaluate
y_pred_lr = pipeline_lr.predict(X_test)

print("=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===")
print(classification_report(y_test, y_pred_lr, target_names=['Fake (0)', 'Real (1)']))

TASK 10: LOGISTIC REGRESSION PIPELINE
Training Logistic Regression model... (This should be much faster!)
Training completed in 125.12 seconds!

=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===
              precision    recall  f1-score   support

    Fake (0)       1.00      1.00      1.00      4669
    Real (1)       1.00      1.00      1.00      4311

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

